# Pengukuran Performa Inferensi

Notebook ini mengukur efisiensi inferensi checkpoint terbaik (fold 1) pada model unimodal dan multimodal, tanpa pelatihan ulang.

| Kelompok | Model |
|---|---|
| Teks | ALBERT-base-v2 |
| Gambar | MobileNetV3-Small, MobileViT-XXS |
| Multimodal | ALBERT + MobileNetV3-Small, ALBERT + MobileViT-XXS |

Metrik: jumlah parameter, ukuran model, FLOPs, latensi (batch 1), throughput (batch 64), dan memori puncak GPU (batch 1).

In [1]:
!pip install transformers timm --quiet

## Konfigurasi

Pengaturan mengikuti notebook pelatihan (FP16, 128 token). Latensi dirata-ratakan dari 100 iterasi setelah 20 iterasi pemanasan, pada GPU Tesla T4.

In [2]:
import os, io, gc, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.flop_counter import FlopCounterMode
from transformers import AlbertModel
import timm
warnings.filterwarnings('ignore')

# ---- Pengaturan pengukuran ----
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True           # sama dengan evaluasi di notebook training (autocast fp16)
MAX_SEQ_LEN = 128        # sama dengan training
BS_LATENCY = 1           # batch size untuk latensi, FLOPs, dan memori
BS_THROUGHPUT = 64       # batch size untuk throughput (sama dengan batch size training)
WARMUP_ITERS = 20
LATENCY_ITERS = 100
THROUGHPUT_ITERS = 30
FOLD = 1                 # fold yang checkpoint-nya dipakai
NUM_CLASSES = 3

# ---- Pengaturan arsitektur (sama dengan notebook training) ----
ALBERT_MODEL = "albert-base-v2"
PROJ_DIM = 256
MLP_DROPOUT = 0.4

# ---- Lokasi pencarian checkpoint (dicari berdasarkan nama file) ----
SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input"]
OUT_CSV = "/kaggle/working/inference_results.csv"

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
else:
    print("Peringatan: GPU tidak terdeteksi. Hasil latensi dan memori tidak sesuai rencana.")
print(f"Torch  : {torch.__version__} | AMP: {USE_AMP}")

Device : cuda
GPU    : Tesla T4
Torch  : 2.11.0+cu128 | AMP: True


## Definisi Model

Kelas model disalin dari notebook pelatihan agar struktur bobot sesuai dengan checkpoint.

In [3]:
# ---- Definisi model (disalin dari notebook training agar nama bobot sama) ----

class TextClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.albert = AlbertModel.from_pretrained(ALBERT_MODEL)
        hidden = self.albert.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, input_ids, attention_mask):
        out = self.albert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.classifier(cls)


class ImageClassifier(nn.Module):
    def __init__(self, model_name, num_classes, img_size=224, dropout=0.3):
        super().__init__()
        self.encoder = timm.create_model(model_name, pretrained=True,
                                         num_classes=0, global_pool='avg')
        self.encoder.eval()
        with torch.no_grad():
            _dummy = torch.zeros(1, 3, img_size, img_size)
            feat_dim = self.encoder(_dummy).shape[-1]
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, num_classes))

    def forward(self, images):
        return self.classifier(self.encoder(images))


class MultimodalConcatModel(nn.Module):
    def __init__(self, image_model_name, num_classes=3, img_size=224):
        super().__init__()
        self.img_size = img_size
        self.text_encoder = AlbertModel.from_pretrained(ALBERT_MODEL)
        text_dim = self.text_encoder.config.hidden_size
        self.image_encoder = timm.create_model(image_model_name, pretrained=True, num_classes=0, global_pool='avg')
        self.image_encoder.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, img_size, img_size)
            feat = self.image_encoder(dummy)
            img_channels = feat.shape[-1]
        self.text_proj = nn.Sequential(nn.Linear(text_dim, PROJ_DIM), nn.LayerNorm(PROJ_DIM), nn.GELU())
        self.img_proj = nn.Sequential(nn.Linear(img_channels, PROJ_DIM), nn.LayerNorm(PROJ_DIM), nn.GELU())
        self.dropout = nn.Dropout(MLP_DROPOUT)
        self.intermediate_mlp = nn.Sequential(nn.Linear(PROJ_DIM*2, PROJ_DIM), nn.LayerNorm(PROJ_DIM), nn.GELU(), nn.Dropout(MLP_DROPOUT))
        self.classifier = nn.Sequential(nn.Linear(PROJ_DIM, 128), nn.ReLU(), nn.Dropout(MLP_DROPOUT), nn.Linear(128, num_classes))

    def forward(self, input_ids, attention_mask, images):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_vec = self.text_proj(text_out.last_hidden_state[:, 0, :])
        img_vec = self.img_proj(self.image_encoder(images))
        fused = torch.cat([text_vec, img_vec], dim=-1)
        fused_rep = self.intermediate_mlp(fused)
        logits = self.classifier(self.dropout(fused_rep))
        return logits


# ---- Daftar model yang diukur ----
# Nama file checkpoint mengikuti nama yang dipakai di notebook training.
# Periksa nama file untuk ALBERT + MobileViT-XXS, lalu sesuaikan bila berbeda.
MODELS = [
    dict(name="ALBERT-base-v2", kind="text", img_size=None,
         build=lambda: TextClassifier(NUM_CLASSES),
         ckpt=f"ALBERT_unimodal_fold{FOLD}.pt"),
    dict(name="MobileNetV3-Small", kind="image", img_size=224,
         build=lambda: ImageClassifier("mobilenetv3_small_100", NUM_CLASSES, img_size=224),
         ckpt=f"MobileNetV3_unimodal_fold{FOLD}.pt"),
    dict(name="MobileViT-XXS", kind="image", img_size=256,
         build=lambda: ImageClassifier("mobilevit_xxs", NUM_CLASSES, img_size=256),
         ckpt=f"MobileViT_unimodal_fold{FOLD}.pt"),
    dict(name="ALBERT + MobileNetV3-Small", kind="multi", img_size=224,
         build=lambda: MultimodalConcatModel("mobilenetv3_small_100", NUM_CLASSES, img_size=224),
         ckpt=f"FINAL_MobileNetV3_AUG_fold{FOLD}.pt"),
    dict(name="ALBERT + MobileViT-XXS", kind="multi", img_size=256,
         build=lambda: MultimodalConcatModel("mobilevit_xxs", NUM_CLASSES, img_size=256),
         ckpt=f"FINAL_MobileViT_AUG_fold{FOLD}.pt"),
]

## Fungsi Pengukuran

Waktu diukur dengan sinkronisasi CUDA. FLOPs dihitung dengan `FlopCounterMode` (fp32) dan bersifat estimasi.

In [4]:
# ---- Fungsi bantu ----

def find_checkpoint(filename):
    for root_dir in SEARCH_ROOTS:
        for root, _, files in os.walk(root_dir):
            if filename in files:
                return os.path.join(root, filename)
    return None


def load_weights(model, filename):
    # Mengembalikan True bila semua bobot berhasil dimuat dari checkpoint.
    path = find_checkpoint(filename)
    if path is None:
        print(f"  [Peringatan] {filename} tidak ditemukan. Memakai bobot awal (parameter, FLOPs, dan latensi tetap valid).")
        return False
    d = torch.load(path, map_location="cpu", weights_only=False)
    state = d.get("model_state_dict", d.get("model_state"))
    res = model.load_state_dict(state, strict=False)
    print(f"  Checkpoint dimuat dari {path}")
    if res.missing_keys:
        print(f"  [Peringatan] Bobot hilang ({len(res.missing_keys)}) contoh {res.missing_keys[:3]}")
    if res.unexpected_keys:
        print(f"  Bobot tidak terpakai ({len(res.unexpected_keys)}) contoh {res.unexpected_keys[:3]}")
    return len(res.missing_keys) == 0


def make_inputs(kind, bs, img_size):
    ids = torch.randint(5, 30000, (bs, MAX_SEQ_LEN), device=DEVICE)
    mask = torch.ones(bs, MAX_SEQ_LEN, dtype=torch.long, device=DEVICE)
    img = torch.randn(bs, 3, img_size or 224, img_size or 224, device=DEVICE)
    if kind == "text":
        return (ids, mask)
    if kind == "image":
        return (img,)
    return (ids, mask, img)


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def forward(model, inputs):
    with torch.autocast(device_type=DEVICE.type, enabled=(USE_AMP and DEVICE.type == "cuda")):
        return model(*inputs)


def measure_latency(model, inputs):
    for _ in range(WARMUP_ITERS):
        forward(model, inputs)
    sync()
    times = []
    for _ in range(LATENCY_ITERS):
        sync()
        t0 = time.perf_counter()
        forward(model, inputs)
        sync()
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))


def measure_throughput(model, inputs, bs):
    for _ in range(WARMUP_ITERS // 2):
        forward(model, inputs)
    sync()
    t0 = time.perf_counter()
    for _ in range(THROUGHPUT_ITERS):
        forward(model, inputs)
    sync()
    elapsed = time.perf_counter() - t0
    return bs * THROUGHPUT_ITERS / elapsed


def measure_peak_memory(model, inputs):
    if DEVICE.type != "cuda":
        return float("nan")
    forward(model, inputs)   # pemanasan agar alokasi awal CUDA tidak ikut terhitung
    sync()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    forward(model, inputs)
    sync()
    return torch.cuda.max_memory_allocated() / 1024**2


def count_gflops(model, inputs):
    # Dihitung dalam fp32 tanpa autocast. Satu perkalian-tambah dihitung sebagai 2 FLOPs.
    with FlopCounterMode(display=False) as fc:
        with torch.no_grad():
            model(*inputs)
    return fc.get_total_flops() / 1e9


def model_size_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1024**2

In [5]:
# ---- Jalankan pengukuran ----
rows = []
for m in MODELS:
    print(f"\n=== {m['name']} ===")
    model = m["build"]()
    loaded = load_weights(model, m["ckpt"])
    model = model.to(DEVICE).eval()

    in_1 = make_inputs(m["kind"], BS_LATENCY, m["img_size"])
    in_n = make_inputs(m["kind"], BS_THROUGHPUT, m["img_size"])

    # Memori diukur lebih dulu agar tidak terpengaruh pengukuran lain.
    peak_mb = measure_peak_memory(model, in_1)
    lat_mean, lat_std = measure_latency(model, in_1)
    thr = measure_throughput(model, in_n, BS_THROUGHPUT)
    gflops = count_gflops(model, in_1)
    n_params = sum(p.numel() for p in model.parameters())
    size_mb = model_size_mb(model)

    rows.append({
        "Model": m["name"],
        "Jenis": {"text": "Unimodal teks", "image": "Unimodal gambar", "multi": "Multimodal"}[m["kind"]],
        "Parameter (juta)": round(n_params / 1e6, 2),
        "Ukuran model (MB)": round(size_mb, 2),
        "GFLOPs": round(gflops, 3),
        "Latensi (ms)": round(lat_mean, 2),
        "Std latensi (ms)": round(lat_std, 2),
        "Throughput (sampel/detik)": round(thr, 1),
        "Memori puncak GPU (MB)": round(peak_mb, 1),
        "Checkpoint dimuat": loaded,
    })

    del model, in_1, in_n
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

result_df = pd.DataFrame(rows)
result_df.to_csv(OUT_CSV, index=False)
print("\n" + result_df.to_string(index=False))
print(f"\nTabel tersimpan di {OUT_CSV}")
print(f"Pengaturan. GPU {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}, "
      f"AMP {USE_AMP}, latensi batch {BS_LATENCY} ({LATENCY_ITERS} iterasi), "
      f"throughput batch {BS_THROUGHPUT}, fold {FOLD}.")


=== ALBERT-base-v2 ===


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 47.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: albert-base-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (9 outputs left out)

  Checkpoint dimuat dari /kaggle/input/datasets/azkurniawan/checkpoint-inferensi/FINAL_MobileNetV3_AUG_fold1.pt

=== ALBERT + MobileViT-XXS ===


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertModel LOAD REPORT from: albert-base-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.dense.weight     | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Checkpoint dimuat dari /kaggle/input/datasets/azkurniawan/checkpoint-inferensi/FINAL_MobileViT_AUG_fold1.pt

                     Model           Jenis  Parameter (juta)  Ukuran model (MB)  GFLOPs  Latensi (ms)  Std latensi (ms)  Throughput (sampel/detik)  Memori puncak GPU (MB)  Checkpoint dimuat
            ALBERT-base-v2   Unimodal teks             11.88              45.33  22.374         10.54              0.40                      383.1                    75.3               True
         MobileNetV3-Small Unimodal gambar              1.78               6.93   0.111          7.31              0.41                     6378.8                    57.5               True
             MobileViT-XXS Unimodal gambar              1.03               4.08   0.814         13.10              0.65                     1688.3                    66.0               True
ALBERT + MobileNetV3-Small      Multimodal             13.83              52.89  22.486         18.24              0.35          

## Hasil dan Pembahasan

- ALBERT mendominasi biaya komputasi (22,374 GFLOPs), sehingga penambahan encoder gambar hanya menaikkan FLOPs 0,5% hingga 3,6%.
- Latensi tidak mengikuti FLOPs. Model multimodal mendekati jumlah latensi kedua encoder penyusunnya (18,24 ms dan 23,89 ms).
- ALBERT + MobileNetV3-Small lebih efisien daripada ALBERT + MobileViT-XXS pada FLOPs, latensi, dan throughput.